```                    
                    ┌─────────────────────────────┐
                    │        DATABRICKS JOB       │
                    │      Notebook Execution     │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │           DRIVER            │
                    │                             │
                    │  SparkSession / SparkContext│
                    │  Catalyst Optimizer         │
                    │  DAG Scheduler              │
                    │  Task Scheduler             │
                    └──────────────┬──────────────┘
                                   │
                                   │ Your Code
                                   ▼
              ┌─────────────────────────────────────────┐
              │         spark.read.csv("10GB.csv")      │
              └────────────────────┬────────────────────┘
                                   │
                                   ▼
                    ┌──────────────────────────┐
                    │     LOGICAL PLAN          │
                    │                          │
                    │ Filter 1                 │
                    │ Filter 2                 │
                    │ Filter 3                 │
                    │ ...                      │
                    │ Filter 10                │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │   CATALYST OPTIMIZER     │
                    │                          │
                    │ Optimize filters         │
                    │ Predicate pushdown        │
                    │ Column pruning            │
                    │ Other optimizations       │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │      PHYSICAL PLAN       │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │       DAG SCHEDULER      │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                         ┌───────────────┐
                         │    STAGE 0    │
                         │   CSV READ    │
                         │   + FILTERS   │
                         └───────┬───────┘
                                 │
                 ┌───────────────┼────────────────┐
                 ▼               ▼                ▼
              Task 0           Task 1           Task 2
                 │               │                │
                 ▼               ▼                ▼
            Partition 0     Partition 1      Partition 2
                 │               │                │
                 ▼               ▼                ▼
             Filter 1        Filter 1         Filter 1
                 ↓               ↓                ↓
             Filter 2        Filter 2         Filter 2
                 ↓               ↓                ↓
                ...             ...              ...
                 ↓               ↓                ↓
             Filter 10       Filter 10        Filter 10
                 │               │                │
                 └───────────────┼────────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │       DELTA WRITE        │
                    │                          │
                    │    Parquet Data Files    │
                    │            +             │
                    │       _delta_log         │
                    └──────────────────────────┘
```